## Parallel Workflows ##

**We always update partial state in paralle workflows where evry node or function return only the updated field or changed field existing are passes as it is.**

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Define State Schema
class BatsMan(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int
    sr: float
    bpb: float
    boundary_percent: float
    summary: str


# Node Functions (Each returning partial state updates)
def calculate_sr(state: BatsMan) -> dict:
    sr = (state["runs"] / state["balls"]) * 100 if state["balls"] > 0 else 0.0
    return {"sr": round(sr, 2)}


def calculate_bpb(state: BatsMan) -> dict:
    total_boundaries = state["fours"] + state["sixes"]
    bpb = state["balls"] / total_boundaries if total_boundaries > 0 else 0.0
    return {"bpb": round(bpb, 2)}


def calculate_boundary_percent(state: BatsMan) -> dict:
    boundary_runs = (state["fours"] * 4) + (state["sixes"] * 6)
    boundary_percent = (boundary_runs / state["runs"]) * 100 if state["runs"] > 0 else 0.0
    return {"boundary_percent": round(boundary_percent, 2)}


def summary(state: BatsMan) -> dict:
    summary_text = f"""
Strike Rate - {state['sr']}
Balls per boundary - {state['bpb']}
Boundary percent - {state['boundary_percent']}%
"""
    return {"summary": summary_text}


# Build Graph
graph = StateGraph(BatsMan)

# Add Nodes
graph.add_node("calculate_sr", calculate_sr)
graph.add_node("calculate_bpb", calculate_bpb)
graph.add_node("calculate_boundary_percent", calculate_boundary_percent)
graph.add_node("summary", summary)

# Add Edges (Fan-out from START to calculations, Fan-in to summary)
graph.add_edge(START, "calculate_sr")
graph.add_edge(START, "calculate_bpb")
graph.add_edge(START, "calculate_boundary_percent")

graph.add_edge("calculate_sr", "summary")
graph.add_edge("calculate_bpb", "summary")
graph.add_edge("calculate_boundary_percent", "summary")

graph.add_edge("summary", END)

# Compile Workflow
workflow = graph.compile()

# Execute Workflow
initial_state = {
    "runs": 85,
    "balls": 42,
    "fours": 8,
    "sixes": 4
}

final_state = workflow.invoke(initial_state)

# Display Output
print("Final State Dictionary:")
print(final_state)

print("\nFormatted Summary:")
print(final_state["summary"])

Final State Dictionary:
{'runs': 85, 'balls': 42, 'fours': 8, 'sixes': 4, 'sr': 202.38, 'bpb': 3.5, 'boundary_percent': 65.88, 'summary': '\nStrike Rate - 202.38\nBalls per boundary - 3.5\nBoundary percent - 65.88%\n'}

Formatted Summary:

Strike Rate - 202.38
Balls per boundary - 3.5
Boundary percent - 65.88%



## LLM Parallel Workflow ##

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.graph import StateGraph
import operator
from typing import TypedDict , Annotated
load_dotenv()



In [ ]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
result = model.invoke("The name of the capital of Pakistan is?")
print(result.content)

In [ ]:
class EssayState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_feedback: Annotated[list[int] ,operator.add]

In [ ]:
graph = StateGraph(EssayState)